# Task 7 — Automated Generative AI Email Assistant with API Functions

Simulates a customer-support workflow: a customer message comes in, an LLM
drafts a professional reply, and the system automatically formats, saves,
and "sends" the email using API-like functions, with error handling for
empty input and failed save/send.

## Objectives checklist
- [x] Load/call an LLM
- [x] Generate structured responses (email format)
- [x] Create API-like functions (save, send, log)
- [x] Automate the pipeline end to end
- [x] Handle simple errors (empty input, failed save)
- [x] Automated tests proving the success path and both failure paths

## About `USE_LIVE_MODEL`
Set the flag below to `True` to download and use a real Hugging Face model
(`google/flan-t5-base`) -- requires internet access. Left `False`, the
notebook uses a template-based offline stand-in generator so the full
pipeline (generate -> format -> save -> send -> log) can be demonstrated
and verified without downloading anything, which is what was used to
produce the outputs saved in this notebook.

In [1]:
%pip install -q transformers torch

Note: you may need to restart the kernel to use updated packages.


## 1. Configuration

In [2]:
USE_LIVE_MODEL = False  # Set True to use a real Hugging Face LLM (requires internet)


## 2. Email generator

Turns a raw customer message into a professional reply.

In [3]:
PROMPT_TEMPLATE = (
    "You are a professional customer support agent. Write a polite, concise "
    "email reply to the following customer message. Include a greeting, an "
    "answer to their concern, and a professional sign-off.\n\n"
    "Customer message: {message}\n\nReply:"
)


def offline_email_llm(prompt: str) -> str:
    """Template-based offline stand-in for a real LLM reply."""
    message = prompt.split("Customer message:", 1)[1].split("\n\nReply:")[0].strip()
    return (
        "Dear Customer,\n\n"
        f"Thank you for reaching out about: \"{message}\"\n\n"
        "We have received your message and our support team is reviewing it now. "
        "We will follow up with a full resolution shortly.\n\n"
        "Best regards,\nCustomer Support Team"
    )


def build_live_generator(model_name: str = "google/flan-t5-base", max_new_tokens: int = 200):
    from transformers import pipeline
    pipe = pipeline("text2text-generation", model=model_name, max_new_tokens=max_new_tokens)
    return lambda prompt: pipe(prompt)[0]["generated_text"].strip()


def get_generator():
    return build_live_generator() if USE_LIVE_MODEL else offline_email_llm


class EmailGenerator:
    def __init__(self, generator=None):
        self.generator = generator or get_generator()

    def generate_reply(self, customer_message: str) -> str:
        if not customer_message or not customer_message.strip():
            raise ValueError("Customer message cannot be empty.")
        prompt = PROMPT_TEMPLATE.format(message=customer_message.strip())
        return self.generator(prompt)


print(f"EmailGenerator ready (USE_LIVE_MODEL={USE_LIVE_MODEL}).")

EmailGenerator ready (USE_LIVE_MODEL=False).


## 3. API-like functions

Simulated backend functions: format, save (to a JSON file), send (simulated), and log (audit trail).

In [4]:
import json
import os
from datetime import datetime, timezone

DATA_DIR = "data"
EMAILS_FILE = os.path.join(DATA_DIR, "sent_emails.json")
LOG_FILE = os.path.join(DATA_DIR, "email_log.txt")


def format_email(recipient: str, subject: str, body: str) -> dict:
    if not body or not body.strip():
        raise ValueError("Email body cannot be empty.")
    return {
        "recipient": recipient or "customer@example.com",
        "subject": subject or "Re: Your inquiry",
        "body": body.strip(),
        "timestamp": datetime.now(timezone.utc).isoformat(),
    }


def save_email(email: dict, emails_file: str = EMAILS_FILE) -> bool:
    try:
        os.makedirs(os.path.dirname(emails_file), exist_ok=True)
        records = []
        if os.path.exists(emails_file):
            with open(emails_file, "r", encoding="utf-8") as f:
                records = json.load(f)
        records.append(email)
        with open(emails_file, "w", encoding="utf-8") as f:
            json.dump(records, f, indent=2)
        return True
    except OSError:
        return False


def send_email(email: dict) -> bool:
    """Simulates sending an email (no real network call)."""
    if not email.get("body"):
        return False
    print(f"[SIMULATED SEND] To: {email['recipient']} | Subject: {email['subject']}")
    return True


def log_action(action: str, details: str = "", log_file: str = LOG_FILE) -> None:
    os.makedirs(os.path.dirname(log_file), exist_ok=True)
    timestamp = datetime.now(timezone.utc).isoformat()
    with open(log_file, "a", encoding="utf-8") as f:
        f.write(f"[{timestamp}] {action}: {details}\n")


print("API-like functions ready.")

API-like functions ready.


## 4. Pipeline

Wires the steps together: generate -> format -> save -> send -> log, short-circuiting with a reported error at whichever stage fails.

In [5]:
def process_customer_message(
    generator: EmailGenerator,
    customer_message: str,
    recipient: str = "",
    emails_file: str = EMAILS_FILE,
    log_file: str = LOG_FILE,
) -> dict:
    result = {"status": "failed", "email": None, "errors": []}

    try:
        reply_body = generator.generate_reply(customer_message)
    except ValueError as e:
        result["errors"].append(str(e))
        log_action("GENERATE_FAILED", str(e), log_file=log_file)
        return result

    try:
        email = format_email(recipient=recipient, subject="Re: Your inquiry", body=reply_body)
    except ValueError as e:
        result["errors"].append(str(e))
        log_action("FORMAT_FAILED", str(e), log_file=log_file)
        return result

    saved = save_email(email, emails_file=emails_file)
    if not saved:
        result["errors"].append("Failed to save email.")
        log_action("SAVE_FAILED", str(email), log_file=log_file)
        return result
    log_action("SAVED", email["subject"], log_file=log_file)

    sent = send_email(email)
    if not sent:
        result["errors"].append("Failed to send email.")
        log_action("SEND_FAILED", str(email), log_file=log_file)
        return result
    log_action("SENT", email["subject"], log_file=log_file)

    result["status"] = "success"
    result["email"] = email
    return result


print("Pipeline ready.")

Pipeline ready.


## 5. Demo run

Simulates a customer message coming in and being handled end to end, plus the empty-input error case.

In [6]:
generator = EmailGenerator()

demo_cases = [
    ("My order hasn\'t arrived yet and it\'s been 2 weeks.", "jane@example.com"),
    ("   ", ""),  # empty input -> should fail gracefully
]

for message, recipient in demo_cases:
    print(f"Customer message: {message!r}")
    result = process_customer_message(generator, message, recipient=recipient)
    if result["status"] == "success":
        email = result["email"]
        print("Status: success")
        print(f"  To: {email['recipient']}")
        print(f"  Subject: {email['subject']}")
        print(f"  Body: {email['body']}")
    else:
        print("Status: failed -- " + "; ".join(result["errors"]))
    print()

Customer message: "My order hasn't arrived yet and it's been 2 weeks."
[SIMULATED SEND] To: jane@example.com | Subject: Re: Your inquiry
Status: success
  To: jane@example.com
  Subject: Re: Your inquiry
  Body: Dear Customer,

Thank you for reaching out about: "My order hasn't arrived yet and it's been 2 weeks."

We have received your message and our support team is reviewing it now. We will follow up with a full resolution shortly.

Best regards,
Customer Support Team

Customer message: '   '
Status: failed -- Customer message cannot be empty.



## 6. Automated tests (offline)

Covers the success path and both required error scenarios: empty customer input, and a failed save (forced by pointing the save path at a location that cannot be created).

In [7]:
import shutil
import tempfile


def test_format_email_rejects_empty_body():
    try:
        format_email("a@b.com", "Subject", "   ")
    except ValueError:
        print("PASS test_format_email_rejects_empty_body")
        return
    raise AssertionError("Expected ValueError for empty body")


def test_send_email_rejects_missing_body():
    assert send_email({"recipient": "a@b.com", "subject": "x", "body": ""}) is False
    print("PASS test_send_email_rejects_missing_body")


def test_pipeline_success_path():
    tmp_dir = tempfile.mkdtemp()
    try:
        emails_file = os.path.join(tmp_dir, "sent_emails.json")
        log_file = os.path.join(tmp_dir, "email_log.txt")
        gen = EmailGenerator(generator=offline_email_llm)
        result = process_customer_message(
            gen, "My order hasn\'t arrived yet.", recipient="jane@example.com",
            emails_file=emails_file, log_file=log_file,
        )
        assert result["status"] == "success", result
        assert result["email"]["recipient"] == "jane@example.com"
        assert os.path.exists(emails_file)
        with open(emails_file) as f:
            assert len(json.load(f)) == 1
        with open(log_file) as f:
            log_contents = f.read()
        assert "SAVED" in log_contents and "SENT" in log_contents
        print("PASS test_pipeline_success_path")
    finally:
        shutil.rmtree(tmp_dir)


def test_pipeline_handles_empty_input():
    tmp_dir = tempfile.mkdtemp()
    try:
        emails_file = os.path.join(tmp_dir, "sent_emails.json")
        log_file = os.path.join(tmp_dir, "email_log.txt")
        gen = EmailGenerator(generator=offline_email_llm)
        result = process_customer_message(gen, "   ", emails_file=emails_file, log_file=log_file)
        assert result["status"] == "failed"
        assert "empty" in result["errors"][0].lower()
        assert not os.path.exists(emails_file)
        with open(log_file) as f:
            assert "GENERATE_FAILED" in f.read()
        print("PASS test_pipeline_handles_empty_input")
    finally:
        shutil.rmtree(tmp_dir)


def test_pipeline_handles_failed_save():
    tmp_dir = tempfile.mkdtemp()
    try:
        blocked_file = os.path.join(tmp_dir, "not_a_dir")
        with open(blocked_file, "w") as f:
            f.write("x")
        emails_file = os.path.join(blocked_file, "sub", "sent_emails.json")
        log_file = os.path.join(tmp_dir, "email_log.txt")
        gen = EmailGenerator(generator=offline_email_llm)
        result = process_customer_message(gen, "Please help with my invoice.", emails_file=emails_file, log_file=log_file)
        assert result["status"] == "failed"
        assert "save" in result["errors"][0].lower()
        with open(log_file) as f:
            assert "SAVE_FAILED" in f.read()
        print("PASS test_pipeline_handles_failed_save")
    finally:
        shutil.rmtree(tmp_dir)


test_format_email_rejects_empty_body()
test_send_email_rejects_missing_body()
test_pipeline_success_path()
test_pipeline_handles_empty_input()
test_pipeline_handles_failed_save()
print("\nAll email assistant tests passed.")

PASS test_format_email_rejects_empty_body
PASS test_send_email_rejects_missing_body
[SIMULATED SEND] To: jane@example.com | Subject: Re: Your inquiry
PASS test_pipeline_success_path
PASS test_pipeline_handles_empty_input
PASS test_pipeline_handles_failed_save

All email assistant tests passed.


## 7. Interactive mode (optional)

Run this cell in a real Jupyter session to type your own customer messages. Exits gracefully if there is no input available (e.g. run non-interactively).

In [8]:
try:
    while True:
        message = input("Customer message (or \'exit\'): ").strip()
        if message.lower() in ("exit", "quit"):
            break
        recipient = input("Customer email (optional): ").strip()
        result = process_customer_message(generator, message, recipient=recipient)
        if result["status"] == "success":
            print(f"Sent: {result['email']}\n")
        else:
            print("Failed: " + "; ".join(result["errors"]) + "\n")
except Exception:
    print("(No interactive input available -- skipping interactive mode.)")

(No interactive input available -- skipping interactive mode.)


## Notes
- Swap `get_generator()` for a real API-based LLM (OpenAI/Anthropic) -- the pipeline logic stays the same.
- Empty customer message -> generation is rejected, nothing is saved or sent, and `GENERATE_FAILED` is logged.
- Save/send failures are caught, logged (`SAVE_FAILED`/`SEND_FAILED`), and reported back to the caller instead of crashing the pipeline.
